# 초급 프로젝트 미션

## 프로젝트 개요

첫번째 프로젝트에 오신 스프린터 여러분들을 환영합니다!

- 지금까지 학습하신 커리큘럼 초반부의 내용들을 토대로, **이미지 인식 기술을 헬스케어 분야에 접목해보는** 프로젝트를 진행해봅시다.
- 여러분들을 헬스케어 스타트업 : 헬스잇(Health Eat) 의 AI 엔지니어링 팀이라고 가정해 볼게요.
    - 헬스잇의 AI 엔지니어링 팀은 유저가 본인의 모바일 애플리케이션으로 자신이 복용중인 약 사진을 찍었을 때, 이미지 인식을 통해 해당 약에 대한 정보를 확인할 수 있는 모델을 만들어야하는 미션을 부여받았습니다.
    - 기업에서는 이를 통해 유저의 건강 상태 및 함께 복용하면 안되는 약 등 헬스케어 정보를 유저들에게 제공함으로써 비즈니스적인 가치를 만들어낼 수 있겠죠.
- 사진 속에 있는 최대 4개의 알약의 이름(클래스)과 위치(바운딩 박스)를 검출하는 모델을 구현하고, 성능을 지속적으로 개선해나가는 것이 프로젝트의 목표입니다.
    
        
- 모델 성능 검증은 코드잇 스프린트에서 별도로 세팅해둔 Kaggle에서 진행됩니다.
    - Submission 가이드라인에 따라 Output을 제출하고 Leaderbord를 통해 모델 성능을 대략적으로 확인해보세요.
    - 순위는 중요하지 않아요. 자신이 속한 팀의 점수가 지속적으로 향상되는지를 확인해보고, 향상되었다면 왜 그랬는지 트래킹하기 위한 수단으로 활용해 봅시다.

In [11]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

In [ ]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import unicodedata  # 0번 섹션에 추가 필요
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 경로 설정
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# Device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# 환경설정

ENV = "local"  # "local" 또는 "colab" 으로만 변경

if ENV == "local":
    BASE_DIR   = "./dataset"
    RUNS_DIR   = "runs/detect"
    FONT_PATH  = "C:/Windows/Fonts/malgun.ttf"        # 윈도우 맑은 고딕
    # FONT_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"  # 리눅스/Mac

elif ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR  = "/content/drive/MyDrive/dataset"
    RUNS_DIR  = "/content/drive/MyDrive/runs/detect"

    # Colab 한글 폰트 설치
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"], capture_output=True)
    FONT_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# ── 한글 폰트 적용 ──────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont(FONT_PATH)
font_name = fm.FontProperties(fname=FONT_PATH).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print(f" 환경: {ENV}")
print(f" 폰트: {font_name}")
print(f" BASE_DIR: {BASE_DIR}")

In [ ]:

# ── 공통 경로 (건드릴 필요 없음) ──────────────────
extract_path           = BASE_DIR
TRAIN_JSON_PATH        = os.path.join(BASE_DIR, "merged_annotations_train_final.json")
TEST_JSON_PATH         = os.path.join(BASE_DIR, "merged_annotations_test_final.json")
TRAIN_IMG_DIR          = os.path.join(BASE_DIR, "train_images")
TEST_IMG_DIR           = os.path.join(BASE_DIR, "test_images")

os.makedirs(PROCESSED_TEST_IMG_DIR, exist_ok=True)

print(f" 환경: {ENV}")
print(f" BASE_DIR: {BASE_DIR}")

In [13]:
############################################################
# 1. 병합된 JSON 파일을 읽어서 DataFrame으로 만들기
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 1) 이미지 정보 매핑 (id -> file_name)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}

    records = []
    # 2) 어노테이션 순회
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        
        if file_name is None: continue
        
        img_path = os.path.join(img_dir, file_name)
        
        # 실제 이미지 파일이 있는지 확인 (선택 사항이지만 안전함)
        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0], # 파일명을 ID로 사용
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    return pd.DataFrame(records)

# 실행
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
print(f"✅ 학습 데이터 로드 완료: {len(df)} 개의 객체 탐지됨")

✅ 학습 데이터 로드 완료: 4526 개의 객체 탐지됨


In [14]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

고유 category_id 개수: 73
num_classes (background 포함): 74


In [15]:
# 파일/폴더 존재 여부 체크
paths_to_check = {
    "TRAIN JSON": TRAIN_JSON_PATH,
    "TEST JSON" : TEST_JSON_PATH,
    "TRAIN IMG DIR": TRAIN_IMG_DIR,
    "TEST IMG DIR" : TEST_IMG_DIR,
}

for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "존재" if exists else "없음"
    print(f"{status} | {name}: {path}")

존재 | TRAIN JSON: ./dataset/merged_annotations_train_final.json
존재 | TEST JSON: ./dataset/merged_annotations_test_final.json
존재 | TRAIN IMG DIR: ./dataset/train_images
존재 | TEST IMG DIR: ./dataset/test_images


In [16]:
# 이미지 파일 개수 확인
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

def count_images(directory):
    """디렉토리 내 이미지 파일 수를 반환"""
    if not os.path.exists(directory):
        return 0
    return sum(
        1 for f in os.listdir(directory)
        if f.lower().endswith(IMG_EXTENSIONS)
    )

n_train = count_images(TRAIN_IMG_DIR)
n_test  = count_images(TEST_IMG_DIR)

print(f"Train 이미지 수 : {n_train:,}")
print(f"Test  이미지 수 : {n_test:,}")
print(f"합계            : {n_train + n_test:,}")

Train 이미지 수 : 1,489
Test  이미지 수 : 843
합계            : 2,332


In [17]:
# 샘플 데이터 구조 출력
with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    train_json = json.load(f)

with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_json = json.load(f)

# JSON 최상위 키 확인
print("Train JSON 최상위 키:", list(train_json.keys()))
print("Test  JSON 최상위 키:", list(test_json.keys()))

Train JSON 최상위 키: ['images', 'annotations', 'categories']
Test  JSON 최상위 키: ['images', 'annotations', 'categories']


In [18]:
for split_name, data in [("Train", train_json), ("Test", test_json)]:
    print(f"[{split_name}]")
    print(f"  이미지 수          : {len(data['images'])}")
    print(f"  어노테이션 수      : {len(data['annotations'])}")
    print(f"  클래스 수 : {len(data['categories'])}")
    print()

[Train]
  이미지 수          : 1489
  어노테이션 수      : 4526
  클래스 수 : 73

[Test]
  이미지 수          : 843
  어노테이션 수      : 1129
  클래스 수 : 73



In [19]:
print("=== images 샘플 ===")
print(json.dumps(train_json['images'][0], ensure_ascii=False, indent=2))

print("\n=== annotations 샘플 ===")
print(json.dumps(train_json['annotations'][0], ensure_ascii=False, indent=2))

print("\n=== categories 샘플 (앞 5개) ===")
for cat in train_json['categories'][:5]:
    print(cat)

=== images 샘플 ===
{
  "file_name": "K-001900-016551-024850-027926_0_2_0_2_90_000_200.png",
  "width": 976,
  "height": 1280,
  "imgfile": "K-001900-016551-024850-027926_0_2_0_2_90_000_200.png",
  "drug_N": "K-001900",
  "drug_S": "정상알약",
  "back_color": "연회색 배경",
  "drug_dir": "앞면",
  "light_color": "주백색",
  "camera_la": 90,
  "camera_lo": 0,
  "size": 200,
  "dl_idx": "1899",
  "dl_mapping_code": "K-001900",
  "dl_name": "보령부스파정 5mg",
  "dl_name_en": "Buspar Tab. 5mg Boryung",
  "img_key": "http://connectdi.com/design/img/drug/1Mxwka5v0lL.jpg",
  "dl_material": "부스피론염산염",
  "dl_material_en": "Buspirone Hydrochloride",
  "dl_custom_shape": "정제, 저작정",
  "dl_company": "보령제약(주)",
  "dl_company_en": "Boryung",
  "di_company_mf": "",
  "di_company_mf_en": "",
  "item_seq": 198700706,
  "di_item_permit_date": "19870323",
  "di_class_no": "[01170]정신신경용제",
  "di_etc_otc_code": "전문의약품",
  "di_edi_code": "641901280,A09302381",
  "chart": "이약은 양면볼록한 장방형의 흰색정제이다",
  "drug_shape": "장방형",
  "thick":

In [20]:
from collections import Counter
# category_id → 이름 매핑
train_id2name = {cat['id']: cat['name'] for cat in train_json['categories']}

# 클래스별 어노테이션 수 집계
cat_counter = Counter(annot['category_id'] for annot in train_json['annotations'])

train_df_class = pd.DataFrame([
    {'category_id': cid, 'name': train_id2name.get(cid, '?'), 'count': cnt}
    for cid, cnt in cat_counter.most_common()
])

print(f"총 클래스 수: {len(train_df_class)}")
print(f"전체 어노테이션 수: {train_df_class['count'].sum():,}")
print(f"\n최다 클래스: {train_df_class.iloc[0]['name']} ({train_df_class.iloc[0]['count']}개)")
print(f"최소 클래스: {train_df_class.iloc[-1]['name']} ({train_df_class.iloc[-1]['count']}개)")
print(f"최대/최소 비율: {train_df_class['count'].max() / train_df_class['count'].min():.1f}x")
print(f"\n상위 10개 클래스:")
print(train_df_class.head(10).to_string(index=False))

총 클래스 수: 73
전체 어노테이션 수: 4,526

최다 클래스: 기넥신에프정(은행엽엑스)(수출용) (514개)
최소 클래스: 브린텔릭스정 20mg (7개)
최대/최소 비율: 73.4x

상위 10개 클래스:
 category_id               name  count
        3482 기넥신에프정(은행엽엑스)(수출용)    514
        3350        일양하이트린정 2mg    240
        1899         보령부스파정 5mg    180
        2482        뮤테란캡슐 100mg    172
       16547        가바토파정 100mg    143
       16550      동아가바펜틴정 800mg    139
       35205       아토젯정 10/40mg    113
       29666           리바로정 4mg    111
       16231          리피토정 20mg    109
       36636       로수젯정10/5밀리그램    108


In [21]:
from collections import Counter
import pandas as pd

# category_id → 이름 매핑
test_id2name = {cat['id']: cat['name'] for cat in test_json['categories']}

# 클래스별 어노테이션 수 집계
test_cat_counter = Counter(annot['category_id'] for annot in test_json['annotations'])

# DataFrame 생성
test_df_class = pd.DataFrame([
    {'category_id': cid, 'name': test_id2name.get(cid, '?'), 'count': cnt}
    for cid, cnt in test_cat_counter.most_common()
])

print(f"총 클래스 수: {len(test_df_class)}")
print(f"전체 어노테이션 수: {test_df_class['count'].sum():,}")
print(f"\n최다 클래스: {test_df_class.iloc[0]['name']} ({test_df_class.iloc[0]['count']}개)")
print(f"최소 클래스: {test_df_class.iloc[-1]['name']} ({test_df_class.iloc[-1]['count']}개)")
print(f"최대/최소 비율: {test_df_class['count'].max() / test_df_class['count'].min():.1f}x")
print(f"\n상위 10개 클래스:")
print(test_df_class.head(10).to_string(index=False))

총 클래스 수: 73
전체 어노테이션 수: 1,129

최다 클래스: 기넥신에프정(은행엽엑스)(수출용) (110개)
최소 클래스: 졸로푸트정 100mg (1개)
최대/최소 비율: 110.0x

상위 10개 클래스:
 category_id               name  count
        3482 기넥신에프정(은행엽엑스)(수출용)    110
        2482        뮤테란캡슐 100mg     53
        3350        일양하이트린정 2mg     52
        1899         보령부스파정 5mg     42
       19860          노바스크정 5mg     36
       16550      동아가바펜틴정 800mg     34
       16547        가바토파정 100mg     31
       16261         크레스토정 20mg     30
       27652       세비카정 10/40mg     29
       25468      아모잘탄정 5/100mg     29


In [22]:
import numpy as np

In [23]:
widths, heights = [], []

for img_info in train_json['images']:
    widths.append(img_info['width'])
    heights.append(img_info['height'])

print(f"Width  - min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height - min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")

# 해상도 조합별 빈도 확인
resolution_counter = Counter(zip(widths, heights))
print(f"\n해상도 종류 수: {len(resolution_counter)}")
print("\n상위 5개 해상도:")
for (w, h), cnt in resolution_counter.most_common(5):
    print(f"  {w}x{h} : {cnt}장")

Width  - min: 976, max: 976, mean: 976
Height - min: 1280, max: 1280, mean: 1280

해상도 종류 수: 1

상위 5개 해상도:
  976x1280 : 1489장


In [24]:
# 이미지당 객체 수 분포
img_annot_count = Counter(annot['image_id'] for annot in train_json['annotations'])
obj_counts = list(img_annot_count.values())

print("이미지당 알약 수 분포:")
for k, v in sorted(Counter(obj_counts).items()):
    print(f"  {k}개 알약: {v:5d}장")

print(f"\n평균 객체 수: {np.mean(obj_counts):.2f}")
print(f"최대 객체 수: {max(obj_counts)}")

이미지당 알약 수 분포:
  1개 알약:    64장
  2개 알약:   302장
  3개 알약:   634장
  4개 알약:   489장

평균 객체 수: 3.04
최대 객체 수: 4


In [25]:
bbox_widths, bbox_heights, bbox_areas = [], [], []

for annot in train_json['annotations']:
    x, y, w, h = annot['bbox']
    bbox_widths.append(w)
    bbox_heights.append(h)
    bbox_areas.append(w * h)

print(f"BBox 너비 - min: {min(bbox_widths):.0f}, max: {max(bbox_widths):.0f}, mean: {np.mean(bbox_widths):.0f}")
print(f"BBox 높이 - min: {min(bbox_heights):.0f}, max: {max(bbox_heights):.0f}, mean: {np.mean(bbox_heights):.0f}")
print(f"BBox 면적 - min: {min(bbox_areas):.0f}, max: {max(bbox_areas):.0f}, mean: {np.mean(bbox_areas):.0f}")

# 이미지 대비 BBox 크기 비율
img_w, img_h = 976, 1280
print(f"\n이미지 대비 BBox 평균 비율:")
print(f"  너비: {np.mean(bbox_widths)/img_w*100:.1f}%")
print(f"  높이: {np.mean(bbox_heights)/img_h*100:.1f}%")

BBox 너비 - min: 125, max: 529, mean: 259
BBox 높이 - min: 123, max: 669, mean: 289
BBox 면적 - min: 18492, max: 272435, mean: 78884

이미지 대비 BBox 평균 비율:
  너비: 26.6%
  높이: 22.6%


In [26]:
invalid_bboxes = []

for annot in train_json['annotations']:
    x, y, w, h = annot['bbox']
    # 너비/높이가 0 이하이거나 이미지 밖으로 나간 경우
    if w <= 0 or h <= 0 or x < 0 or y < 0 or x + w > 976 or y + h > 1280:
        invalid_bboxes.append(annot['id'])

print(f"비정상 BBox 수: {len(invalid_bboxes)}")

비정상 BBox 수: 2


In [27]:
for annot in train_json['annotations']:
    x, y, w, h = annot['bbox']
    if w <= 0 or h <= 0 or x < 0 or y < 0 or x + w > 976 or y + h > 1280:
        print(f"annotation id : {annot['id']}")
        print(f"image_id      : {annot['image_id']}")
        print(f"bbox          : {annot['bbox']}")
        print(f"x+w={x+w:.1f} (max 976), y+h={y+h:.1f} (max 1280)")
        print()

annotation id : 903
image_id      : 239
bbox          : [6567, 625, 311, 315]
x+w=6878.0 (max 976), y+h=940.0 (max 1280)

annotation id : 1157
image_id      : 310
bbox          : [653, 8889, 217, 217]
x+w=870.0 (max 976), y+h=9106.0 (max 1280)



In [28]:
from collections import defaultdict
import matplotlib.patches as patches

id2filename = {img['id']: img['file_name'] for img in train_json['images']}

img2annots = defaultdict(list)
for annot in train_json['annotations']:
    img2annots[annot['image_id']].append(annot)

def visualize_sample(image_id):
    fname = id2filename.get(image_id)
    img_path = os.path.join(TRAIN_IMG_DIR, os.path.basename(fname))
    img = Image.open(img_path).convert('RGB')

    fig, ax = plt.subplots(1, figsize=(6, 8))
    ax.imshow(img)

    colors = ['red', 'blue', 'green', 'orange']
    for i, annot in enumerate(img2annots[image_id]):
        x, y, w, h = annot['bbox']
        color = colors[i % len(colors)]
        rect = patches.Rectangle((x, y), w, h,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        label = train_id2name.get(annot['category_id'], '?')
        ax.text(x, y - 5, label, color=color, fontsize=8,
                bbox=dict(facecolor='white', alpha=0.6, pad=1))

    ax.set_title(f"image_id: {image_id}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# 알약 수별로 샘플 1장씩 시각화
for n_obj in [1, 2, 3, 4]:
    sample_id = [img_id for img_id, cnt in img_annot_count.items() if cnt == n_obj][0]
    print(f"--- 알약 {n_obj}개 샘플 ---")
    visualize_sample(sample_id)

--- 알약 1개 샘플 ---


<Figure size 600x800 with 1 Axes>

--- 알약 2개 샘플 ---


<Figure size 600x800 with 1 Axes>

--- 알약 3개 샘플 ---


<Figure size 600x800 with 1 Axes>

--- 알약 4개 샘플 ---


<Figure size 600x800 with 1 Axes>

In [29]:
# 라벨링 안 된 알약이 얼마나 되는지 파악하기 위해
# 어노테이션 수 분포를 다시 한번 확인
print("어노테이션 기준 이미지당 알약 수:")
for k, v in sorted(Counter(obj_counts).items()):
    print(f"  라벨 {k}개: {v:5d}장 ({v/len(obj_counts)*100:.1f}%)")

어노테이션 기준 이미지당 알약 수:
  라벨 1개:    64장 (4.3%)
  라벨 2개:   302장 (20.3%)
  라벨 3개:   634장 (42.6%)
  라벨 4개:   489장 (32.8%)


In [30]:
# category_id 재매핑 (불연속 id → 0부터 연속 정수)
original_ids = sorted([cat['id'] for cat in train_json['categories']])
id_remap = {old_id: new_id for new_id, old_id in enumerate(original_ids)}

# categories 업데이트
for cat in train_json['categories']:
    cat['id'] = id_remap[cat['id']]

# annotations 업데이트
for annot in train_json['annotations']:
    annot['category_id'] = id_remap[annot['category_id']]

# 확인
print(f"총 클래스 수: {len(id_remap)}")
print("\n재매핑 샘플 (앞 5개):")
for old, new in list(id_remap.items())[:5]:
    print(f"  {old:6d} → {new}")

총 클래스 수: 73

재매핑 샘플 (앞 5개):
    1899 → 0
    2482 → 1
    3350 → 2
    3482 → 3
    3543 → 4


In [31]:
# 양방향 매핑 저장
new_id2name = {cat['id']: cat['name'] for cat in train_json['categories']}
new_id2old  = {new_id: old_id for old_id, new_id in id_remap.items()}
old_id2new  = id_remap  # 이미 만들어둔 것

print("새 id → 약 이름:")
for i in range(5):
    print(f"  {i} → {new_id2name[i]}")

print("\n새 id → 원래 id:")
for i in range(5):
    print(f"  {i} → {new_id2old[i]}")

새 id → 약 이름:
  0 → 보령부스파정 5mg
  1 → 뮤테란캡슐 100mg
  2 → 일양하이트린정 2mg
  3 → 기넥신에프정(은행엽엑스)(수출용)
  4 → 무코스타정(레바미피드)(비매품)

새 id → 원래 id:
  0 → 1899
  1 → 2482
  2 → 3350
  3 → 3482
  4 → 3543


In [32]:
from PIL import Image, ImageOps
import os

def letterbox(image, target_size=640, fill_color=(114, 114, 114)):
    """
    비율을 유지하며 target_size x target_size 로 변환
    빈 공간은 fill_color 로 채움 (YOLO 기본값: 회색 114)
    반환값: 패딩된 이미지, (scale, pad_w, pad_h)
    """
    orig_w, orig_h = image.size
    scale = target_size / max(orig_w, orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)

    image = image.resize((new_w, new_h), Image.BILINEAR)

    # 좌우/상하 패딩 계산
    pad_w = (target_size - new_w) // 2
    pad_h = (target_size - new_h) // 2

    image = ImageOps.expand(image, (pad_w, pad_h, target_size - new_w - pad_w, target_size - new_h - pad_h), fill=fill_color)

    return image, (scale, pad_w, pad_h)

def transform_bbox(bbox, scale, pad_w, pad_h):
    """
    원본 BBox 좌표를 letterbox 변환에 맞게 조정
    bbox: [x, y, w, h] (COCO 포맷)
    """
    x, y, w, h = bbox
    x = x * scale + pad_w
    y = y * scale + pad_h
    w = w * scale
    h = h * scale
    return [x, y, w, h]

# 샘플 1장으로 확인
sample_img_info = train_json['images'][0]
sample_path = os.path.join(TRAIN_IMG_DIR, os.path.basename(sample_img_info['file_name']))

orig_img = Image.open(sample_path).convert('RGB')
resized_img, (scale, pad_w, pad_h) = letterbox(orig_img)

print(f"원본 크기  : {orig_img.size}")
print(f"변환 후 크기: {resized_img.size}")
print(f"scale={scale:.4f}, pad_w={pad_w}, pad_h={pad_h}")

원본 크기  : (976, 1280)
변환 후 크기: (640, 640)
scale=0.5000, pad_w=76, pad_h=0


In [33]:
import matplotlib.patches as patches

sample_annots = [a for a in train_json['annotations'] if a['image_id'] == sample_img_info['id']]

fig, axes = plt.subplots(1, 2, figsize=(12, 7))

# 원본
axes[0].imshow(orig_img)
axes[0].set_title(f"원본 {orig_img.size}")
for annot in sample_annots:
    x, y, w, h = annot['bbox']
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    axes[0].add_patch(rect)

# letterbox 변환 후
axes[1].imshow(resized_img)
axes[1].set_title(f"letterbox 후 {resized_img.size}")
for annot in sample_annots:
    x, y, w, h = transform_bbox(annot['bbox'], scale, pad_w, pad_h)
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    axes[1].add_patch(rect)

plt.tight_layout()
plt.show()

<Figure size 1200x700 with 2 Axes>

In [34]:
import os

PROCESSED_TRAIN_IMG_DIR = os.path.join(extract_path, "processed_train_images")
os.makedirs(PROCESSED_TRAIN_IMG_DIR, exist_ok=True)

# 전체 이미지에 letterbox 적용 후 저장
scale_info = {}  # image_id → (scale, pad_w, pad_h) 저장

for img_info in train_json['images']:
    img_path = os.path.join(TRAIN_IMG_DIR, os.path.basename(img_info['file_name']))
    save_path = os.path.join(PROCESSED_TRAIN_IMG_DIR, os.path.basename(img_info['file_name']))

    img = Image.open(img_path).convert('RGB')
    resized, (scale, pad_w, pad_h) = letterbox(img)
    resized.save(save_path)

    scale_info[img_info['id']] = (scale, pad_w, pad_h)

print(f"저장 완료: {len(scale_info)}장 → {PROCESSED_TRAIN_IMG_DIR}")

저장 완료: 1489장 → ./dataset/processed_train_images


In [ ]:
import os
from PIL import Image

TEST_IMG_DIR           = os.path.join(BASE_DIR, "test_images")
PROCESSED_TEST_IMG_DIR = os.path.join(BASE_DIR, "processed_test_images")

os.makedirs(PROCESSED_TEST_IMG_DIR, exist_ok=True)  # 디렉토리 생성

count = 0

for filename in os.listdir(TEST_IMG_DIR):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        file_path = os.path.join(TEST_IMG_DIR, filename)
        try:
            img = Image.open(file_path).convert("RGB")
            resized, (scale, pad_w, pad_h) = letterbox(img)  # letterbox 함수 확인 필요

            save_path = os.path.join(PROCESSED_TEST_IMG_DIR, filename)
            resized.save(save_path)
            count += 1
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"저장 완료: {count}장 → {PROCESSED_TEST_IMG_DIR}")

저장 완료: 843장 → ./dataset/processed_test_images


# Yolo 모델

## 모델 학습 준비

In [ ]:
import os


data_yaml = {
    "path": YOLO_BASE,
    "train": "images/train",
    "val": "images/val",
    "nc": len(new_id2name),
    "names": [new_id2name[i] for i in range(len(new_id2name))]
}

with open(YOLO_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True)
for p in paths:
    os.makedirs(os.path.join(YOLO_BASE, p), exist_ok=True)

print("YOLO 폴더 구조 생성 완료")

YOLO 폴더 구조 생성 완료


In [37]:
from PIL import Image
import os

YOLO_IMG_TRAIN = os.path.join(YOLO_BASE, "images/train")
YOLO_LBL_TRAIN = os.path.join(YOLO_BASE, "labels/train")

IMG_SIZE = 640  # letterbox 크기

for img_info in train_json['images']:
    img_id = img_info['id']
    file_name = os.path.basename(img_info['file_name'])

    src_img_path = os.path.join(PROCESSED_TRAIN_IMG_DIR, file_name)
    dst_img_path = os.path.join(YOLO_IMG_TRAIN, file_name)

    # 이미지 복사
    Image.open(src_img_path).save(dst_img_path)

    # 해당 이미지 annotation 가져오기
    annots = [a for a in train_json['annotations'] if a['image_id'] == img_id]

    scale, pad_w, pad_h = scale_info[img_id]

    label_path = os.path.join(YOLO_LBL_TRAIN, file_name.replace(".jpg", ".txt").replace(".png", ".txt"))

    with open(label_path, 'w') as f:
        for ann in annots:
            x, y, w, h = ann['bbox']
            cls = ann['category_id']  # 이미 remap된 상태면 그대로 사용

            # letterbox 반영
            x = x * scale + pad_w
            y = y * scale + pad_h
            w = w * scale
            h = h * scale

            # YOLO format
            x_center = (x + w / 2) / IMG_SIZE
            y_center = (y + h / 2) / IMG_SIZE
            w /= IMG_SIZE
            h /= IMG_SIZE

            # clamp (안정성)
            x_center = min(max(x_center, 0), 1)
            y_center = min(max(y_center, 0), 1)
            w = min(max(w, 0), 1)
            h = min(max(h, 0), 1)

            f.write(f"{cls} {x_center} {y_center} {w} {h}\n")

print("YOLO 라벨 생성 완료")

YOLO 라벨 생성 완료


In [38]:
import random
import shutil

random.seed(42)

train_img_dir = os.path.join(YOLO_BASE, "images/train")
train_lbl_dir = os.path.join(YOLO_BASE, "labels/train")

val_img_dir = os.path.join(YOLO_BASE, "images/val")
val_lbl_dir = os.path.join(YOLO_BASE, "labels/val")

images = [f for f in os.listdir(train_img_dir) if f.endswith((".jpg", ".png"))]

random.shuffle(images)

split_idx = int(len(images) * 0.8)
val_images = images[split_idx:]

for img_name in val_images:
    lbl_name = img_name.replace(".jpg", ".txt").replace(".png", ".txt")

    # move (여기서는 move OK)
    shutil.move(os.path.join(train_img_dir, img_name),
                os.path.join(val_img_dir, img_name))

    shutil.move(os.path.join(train_lbl_dir, lbl_name),
                os.path.join(val_lbl_dir, lbl_name))

print("Train/Val split 완료")

Train/Val split 완료


In [ ]:
import yaml

data_yaml = {
    "path": YOLO_BASE,
    "train": "images/train",
    "val": "images/val",
    "nc": len(new_id2name),
    "names": [new_id2name[i] for i in range(len(new_id2name))]
}

with open(YOLO_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True)

print("data.yaml 생성 완료")

data.yaml 생성 완료


## 모델 학습

In [63]:
from ultralytics import YOLO

model_yolo8m = YOLO("yolov8m.pt")

model_yolo8m.train(
    data=YOLO_YAML_PATH,,
    epochs=40,          # 학습 반복 횟수

    imgsz=640,          # 입력 이미지 크기
    batch=8,            # 배치 크기

    workers=2,          # 데이터 로딩 병렬 스레드 수
    cache='disk',       # 데이터 캐시 방식 (False / 'ram' / 'disk')

    amp=True,           # 혼합 정밀도 학습 (메모리 절약 + 속도 향상)
    optimizer="AdamW",  # 옵티마이저 종류
    cos_lr=True,        # 코사인 학습률 스케줄러 사용 여부
    patience=10,        # Early stopping 기준 에폭 수

    mosaic=1.0,         # 모자이크 증강 확률 (이미지 4장 합성)
    warmup_epochs=3.0,  # 학습률 워밍업 에폭 수

    iou=0.6,            # NMS IoU 임계값
    scale=0.5,          # 이미지 스케일 증강 범위

    mixup=0.05,         # Mixup 증강 확률 (이미지 2장 혼합)
    copy_paste=0.1,     # Copy-Paste 증강 확률 (객체 복사 붙여넣기)

    hsv_h=0.015,        # 색조(Hue) 증강 강도
    hsv_s=0.5,          # 채도(Saturation) 증강 강도
    hsv_v=0.4,          # 명도(Value) 증강 강도

    flipud=0.2,         # 상하 반전 증강 확률
    fliplr=0.5,         # 좌우 반전 증강 확률

    cls=0.4             # 분류 손실 가중치
)

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.4, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./dataset/yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.6, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000023CA97A8A90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026, 

In [65]:
import os

base = "runs/detect"

for root, dirs, files in os.walk(base):
    for file in files:
        if file.endswith(".pt"):
            print(os.path.join(root, file))

runs/detect\train\weights\best.pt
runs/detect\train\weights\last.pt
runs/detect\train2\weights\best.pt
runs/detect\train2\weights\epoch0.pt
runs/detect\train2\weights\epoch10.pt
runs/detect\train2\weights\epoch15.pt
runs/detect\train2\weights\epoch20.pt
runs/detect\train2\weights\epoch25.pt
runs/detect\train2\weights\epoch30.pt
runs/detect\train2\weights\epoch35.pt
runs/detect\train2\weights\epoch5.pt
runs/detect\train2\weights\last.pt
runs/detect\train6\weights\best.pt
runs/detect\train6\weights\last.pt
runs/detect\train7\weights\best.pt
runs/detect\train7\weights\last.pt
runs/detect\train8\weights\best.pt
runs/detect\train8\weights\last.pt
runs/detect\train9\weights\best.pt
runs/detect\train9\weights\last.pt


In [66]:
import pandas as pd
import os

base = "runs/detect"
results = []

for d in os.listdir(base):
    csv_path = os.path.join(base, d, "results.csv")
    if not os.path.exists(csv_path):
        continue

    df = pd.read_csv(csv_path)

    # 가능한 컬럼 후보들
    candidates = [
        "metrics/mAP50-95",
        "metrics/mAP50-95(B)",
        "metrics/mAP50",
        "metrics/mAP50(B)",
        "map",
        "map50"
    ]

    found_col = None
    for c in candidates:
        if c in df.columns:
            found_col = c
            break

    if found_col is None:
        print(f"{d}: mAP 컬럼 없음 → skip")
        continue

    best_map = df[found_col].max()
    results.append((d, best_map, found_col))

# 정렬
results = sorted(results, key=lambda x: x[1], reverse=True)

for r in results:
    print(r)

('train7', np.float64(0.86549), 'metrics/mAP50-95(B)')
('train9', np.float64(0.85023), 'metrics/mAP50-95(B)')
('train6', np.float64(0.8377), 'metrics/mAP50-95(B)')
('train8', np.float64(0.82431), 'metrics/mAP50-95(B)')
('train2', np.float64(0.7981), 'metrics/mAP50-95(B)')
('train', np.float64(0.73184), 'metrics/mAP50-95(B)')


## 모델 추론 및 csv 파일 생성

In [64]:
import os
import numpy as np
import cv2
from ensemble_boxes import weighted_boxes_fusion

rows = []
annotation_id = 1


# 모델 경로 (성능 순)


MODEL_PATHS = [
    os.path.join(RUNS_DIR, "train7/weights/best.pt"),
    os.path.join(RUNS_DIR, "train6/weights/best.pt"),
    os.path.join(RUNS_DIR, "train8/weights/best.pt"),
]
yolo_model_paths = [
    "runs/detect/train7/weights/best.pt",  # mAP 0.865 (1위)
    "runs/detect/train6/weights/best.pt",  # mAP 0.838 (2위)
    "runs/detect/train8/weights/best.pt"   # mAP 0.824 (3위)
]

yolo_models = [YOLO(p) for p in yolo_model_paths]

# 성능 차이 반영한 모델별 가중치
weights = [1.5, 1.1, 1.0] 

# 설정값
wbf_score_threshold = 0.05
final_score_threshold = 0.3
iou_threshold = 0.55


# 테스트 이미지 목록
test_files = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith((".jpg", ".png"))])

for f in test_files:
    img_path = os.path.join(TEST_IMG_DIR, f)
    image_id = os.path.splitext(f)[0]

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    boxes_list = []
    scores_list = []
    labels_list = []

    # 모델별로 독립된 결과 생성
    for model_idx, yolo_model in enumerate(yolo_models):

        model_boxes = []
        model_scores = []
        model_labels = []

        # 원본 추론
        results = yolo_model(img, verbose=False)[0]

        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                conf = float(box.conf.item())
                if conf < wbf_score_threshold:
                    continue

                cls = int(box.cls.item())
                if cls not in new_id2old:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                model_boxes.append([x1 / w, y1 / h, x2 / w, y2 / h])
                model_scores.append(conf)
                model_labels.append(cls)

        # flip TTA (같은 모델 안에 포함)
        img_flip = cv2.flip(img, 1)
        results_flip = yolo_model(img_flip, verbose=False)[0]

        if results_flip.boxes is not None and len(results_flip.boxes) > 0:
            for box in results_flip.boxes:
                conf = float(box.conf.item())
                if conf < wbf_score_threshold:
                    continue

                cls = int(box.cls.item())
                if cls not in new_id2old:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                # flip 복원
                x1_f = w - x2
                x2_f = w - x1

                model_boxes.append([x1_f / w, y1 / h, x2_f / w, y2 / h])
                model_scores.append(conf)
                model_labels.append(cls)

        # 모델 단위로 append
        if len(model_boxes) > 0:
            boxes_list.append(model_boxes)
            scores_list.append(model_scores)
            labels_list.append(model_labels)

    # 탐지 실패 시 skip
    if len(boxes_list) == 0:
        continue

    # WBF (모델별 weight 반영됨)
    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list,
        scores_list,
        labels_list,
        weights=weights,
        iou_thr=iou_threshold,
        skip_box_thr=wbf_score_threshold
    )

    # 결과 정리
    for i in range(len(boxes)):
        score = float(scores[i])
        if score < final_score_threshold:
            continue

        x1 = boxes[i][0] * w
        y1 = boxes[i][1] * h
        x2 = boxes[i][2] * w
        y2 = boxes[i][3] * h

        w_box = max(0, x2 - x1)
        h_box = max(0, y2 - y1)

        if w_box < 2 or h_box < 2:
            continue

        orig_cat = new_id2old[int(labels[i])]

        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": orig_cat + 1,
            "bbox_x": float(x1),
            "bbox_y": float(y1),
            "bbox_w": float(w_box),
            "bbox_h": float(h_box),
            "score": score,
        })

        annotation_id += 1

print(f"추론 완료 - 총 {len(rows)}개 객체 감지")

NameError: name 'RUNS_DIR' is not defined

In [61]:
# DataFrame 생성
df_sub = pd.DataFrame(rows, columns=[
    "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])

# 이미지 ID별 점수 높은 순 정렬 후 상위 4개 추출
df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
df_sub = df_sub.groupby("image_id").head(4).reset_index(drop=True)

# annotation_id 재부여
df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

# 저장
output_path = os.path.join(extract_path, "ensemble_submission.csv")
df_sub.to_csv(output_path, index=False)

print(f"✅ 파일 생성 완료: {output_path}")
print(f"📊 총 예측 객체 수: {len(df_sub)}")
print(df_sub.head())

✅ 파일 생성 완료: ./dataset/ensemble_submission.csv
📊 총 예측 객체 수: 3238
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1        27926  598.686585  671.898651  256.456827   
1              2        1         1900  157.365670  251.844273  203.976007   
2              3        1        16551  556.702779   69.009638  394.210673   
3              4        1        24850  173.220361  742.306442  178.004456   
4              5       10        16548  100.923904  808.053970  242.239025   

       bbox_h     score  
0  482.331619  0.809093  
1  123.919125  0.797461  
2  407.851667  0.777017  
3  289.211197  0.586149  
4  237.341919  0.872027  


## 모델 검증